# 03 - EDA & Visualization

Notebook ini menjawab lima business question HealPoint menggunakan visualisasi dan explanatory analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/appointments_clean.csv')
sns.set_theme(style='whitegrid')

## BQ1 - Berapa tingkat no-show overall?

No-show rate keseluruhan adalah **20.2%**. Ini berarti sekitar satu dari lima appointment berisiko tidak dihadiri pasien.

In [ ]:
counts = df['is_no_show'].value_counts().rename(index={0: 'Show', 1: 'No-show'})
counts.plot(kind='pie', autopct='%1.1f%%', ylabel='', title='Overall Appointment Attendance')
plt.show()

**Kesimpulan BQ1:** no-show adalah masalah operasional yang signifikan dan layak menjadi target fitur AI HealPoint.

## BQ2 - Apakah waiting days memengaruhi no-show?

Appointment di hari yang sama memiliki no-show rate **4.6%**, sedangkan appointment dengan waiting days lebih dari 30 hari memiliki no-show rate **33.0%**.

In [ ]:
wait_bins = pd.cut(df['waiting_days'], bins=[-1, 0, 3, 7, 14, 30, 180], labels=['Same day', '1-3', '4-7', '8-14', '15-30', '>30'])
wait_rate = df.assign(wait_bin=wait_bins).groupby('wait_bin', observed=True)['is_no_show'].mean()
wait_rate.plot(kind='bar', title='No-show Rate by Waiting Days')
plt.ylabel('No-show Rate')
plt.show()

In [ ]:
df[['waiting_days', 'is_no_show']].corr()

**Kesimpulan BQ2:** semakin lama jarak jadwal dengan hari appointment, risiko no-show cenderung meningkat. Fitur reminder dan rekomendasi jadwal menjadi relevan.

## BQ3 - Kelompok usia mana yang berisiko lebih tinggi?

Kelompok usia dengan no-show rate tertinggi adalah **teen** dengan rate **26.6%**.

In [ ]:
age_rate = df.groupby('age_group')['is_no_show'].mean().sort_values(ascending=False)
age_rate.plot(kind='bar', title='No-show Rate by Age Group')
plt.ylabel('No-show Rate')
plt.show()

**Kesimpulan BQ3:** segmentasi usia dapat digunakan untuk prioritas reminder dan pendekatan komunikasi yang berbeda.

## BQ4 - Wilayah mana yang perlu diprioritaskan?

Wilayah prioritas tertinggi berdasarkan minimal 100 appointment adalah **Santos Dumont** dengan **1,276** appointment dan no-show rate **28.9%**.

In [ ]:
top_area = (df.groupby('neighbourhood')
    .agg(total=('appointment_id', 'count'), no_show_rate=('is_no_show', 'mean'))
    .query('total >= 100')
    .sort_values('no_show_rate', ascending=False)
    .head(10))
top_area['no_show_rate'].sort_values().plot(kind='barh', title='Top 10 Neighbourhood by No-show Rate')
plt.xlabel('No-show Rate')
plt.show()

**Kesimpulan BQ4:** wilayah dengan volume cukup besar dan no-show rate tinggi dapat menjadi target intervensi operasional.

## BQ5 - Apakah SMS reminder berkaitan dengan no-show?

No-show rate pasien tanpa SMS adalah **16.7%**, sedangkan pasien yang menerima SMS adalah **27.6%**. Interpretasi harus hati-hati karena penerima SMS mungkin berasal dari appointment dengan waiting days lebih panjang.

In [ ]:
sms_rate = df.groupby('sms_received')['is_no_show'].mean().rename(index={0: 'No SMS', 1: 'SMS Received'})
sms_rate.plot(kind='bar', title='No-show Rate by SMS Reminder Status')
plt.ylabel('No-show Rate')
plt.show()

**Kesimpulan BQ5:** SMS perlu dianalisis bersama fitur lain seperti waiting days. SMS bukan satu-satunya faktor, tetapi tetap penting sebagai bagian dari reminder strategy.

## Analisis Tambahan - Korelasi Fitur Numerik

In [ ]:
numeric_cols = ['age', 'waiting_days', 'scheduled_hour', 'scholarship', 'hypertension', 'diabetes', 'alcoholism', 'handicap', 'sms_received', 'has_chronic_condition', 'is_no_show']
plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_cols].corr(), cmap='Blues', annot=False)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
df['waiting_days'].clip(upper=60).plot(kind='hist', bins=30, title='Waiting Days Distribution (clipped at 60)')
plt.xlabel('Waiting Days')
plt.show()

## Kesimpulan Umum EDA

Fitur `waiting_days`, `age_group`, `sms_received`, dan `neighbourhood` layak digunakan dalam model prediksi no-show karena memiliki hubungan bisnis dan variasi pola terhadap target.